# Tugas 1 - Milestone 3: Analisis Temporal dan Advanced EDA

**Mata Kuliah:** Analisis Big Data  
**Nama Mahasiswa:** Mohammad Ravlindo Saputra  
**NIM:** 202310370311115  
**Username GitHub:** Ravlindo

Notebook ini menggunakan dataset bersih dari Milestone 2 untuk menganalisis pola publikasi berita Indonesia. Pengerjaan dibagi menjadi tiga tahap:

1. inisialisasi dan analisis temporal/window;
2. enam visualisasi interaktif;
3. lima insight analitik dan validasi akhir.

## Bagian 1: Inisialisasi dan Memuat Data Bersih

In [14]:
from pathlib import Path

import duckdb
import plotly.express as px
import plotly.graph_objects as go
import polars as pl

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)

print(f"Polars {pl.__version__} | DuckDB {duckdb.__version__}")

Polars 1.44.2 | DuckDB 1.5.6


In [15]:
cwd = Path.cwd().resolve()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Root project tidak ditemukan.")

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_bersih.parquet"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset bersih tidak ditemukan: {DATA_PATH}. "
        "Jalankan 02_data_cleaning.ipynb terlebih dahulu."
    )

print(f"Dataset: {DATA_PATH}")
print(f"Ukuran file: {DATA_PATH.stat().st_size / (1024 ** 2):,.2f} MB")

Dataset: C:\Users\endo1\Documents\SEMESTER  7\big data\tugas1-MohammadRavlindoS\data\processed\dataset_bersih.parquet
Ukuran file: 691.27 MB


In [16]:
df = pl.scan_parquet(DATA_PATH)
schema = df.collect_schema()
required_columns = {
    "date", "link", "title", "text", "source_domain",
    "title_length", "text_length",
}
missing_columns = required_columns - set(schema.names())
if missing_columns:
    raise KeyError(f"Kolom wajib tidak ditemukan: {sorted(missing_columns)}")

data_summary = df.select(
    pl.len().alias("jumlah_baris"),
    pl.col("date").min().alias("tanggal_awal"),
    pl.col("date").max().alias("tanggal_akhir"),
    pl.col("source_domain").n_unique().alias("jumlah_domain"),
).collect()

print(f"Data loaded: {data_summary.item(0, 'jumlah_baris'):,} baris x {len(schema)} kolom")
data_summary

Data loaded: 928,082 baris x 11 kolom


jumlah_baris,tanggal_awal,tanggal_akhir,jumlah_domain
u32,date,date,u32
928082,2019-05-10,2023-10-01,175


In [17]:
df.head(5).collect()

date,link,title,text,title_length,text_length,title_length_outlier,text_length_outlier,title_length_clean,text_length_clean,source_domain
date,str,str,str,u32,u32,bool,bool,u32,u32,str
2019-05-10,"""https://www.cnbcindonesia.com/…","""Potret Jemaah Haji Mulai Melak…","""Haji tahun ini merupakan tanta…",57,438,false,false,57,438,"""cnbcindonesia.com"""
2019-08-19,"""https://www.cnbcindonesia.com/…","""Cek Warga China Mencoba Robot …","""Seorang anggota staf mengambil…",61,460,false,false,61,460,"""cnbcindonesia.com"""
2019-08-26,"""https://www.cnbcindonesia.com/…","""Ajaib! Tato Elektronik Buatan …","""""Di masa depan, apa yang kami …",58,327,false,false,58,327,"""cnbcindonesia.com"""
2019-10-30,"""https://www.cnbcindonesia.com/…","""Chairul Tanjung Hadiri BSI GIF…","""Menurutnya, karakter anak-anak…",36,601,false,false,36,601,"""cnbcindonesia.com"""
2022-03-28,"""https://www.cnbcindonesia.com/…","""11 Tanda Rumah Anda Sudah Terc…","""Jakarta, CNBC Indonesia - Polu…",60,2114,false,false,60,2114,"""cnbcindonesia.com"""


## Bagian 2: Analisis Temporal dan Window Functions

Artikel diagregasi per bulan menggunakan `group_by_dynamic()`. Selanjutnya digunakan `rolling_mean()` untuk meredam fluktuasi jangka pendek dan `shift()` untuk menghitung perubahan jumlah artikel dibandingkan bulan sebelumnya.

In [18]:
monthly_lf = (
    df
    .sort("date")
    .group_by_dynamic("date", every="1mo")
    .agg(
        pl.len().cast(pl.Int64).alias("jumlah_artikel"),
        pl.col("source_domain").n_unique().alias("domain_aktif"),
        pl.col("title_length").mean().round(2).alias("rata_panjang_judul"),
        pl.col("text_length").mean().round(2).alias("rata_panjang_teks"),
    )
    .sort("date")
)

monthly = (
    monthly_lf
    .with_columns(
        pl.col("jumlah_artikel").rolling_mean(window_size=3).alias("moving_average_3_bulan"),
        pl.col("jumlah_artikel").shift(1).alias("artikel_bulan_sebelumnya"),
    )
    .with_columns(
        ((pl.col("jumlah_artikel") - pl.col("artikel_bulan_sebelumnya"))
         / pl.col("artikel_bulan_sebelumnya") * 100)
        .round(2)
        .alias("pertumbuhan_persen")
    )
    .collect()
)
monthly

date,jumlah_artikel,domain_aktif,rata_panjang_judul,rata_panjang_teks,moving_average_3_bulan,artikel_bulan_sebelumnya,pertumbuhan_persen
date,i64,u32,f64,f64,f64,i64,f64
2019-05-01,1,1,57.0,438.0,null,null,null
2019-08-01,2,1,59.5,393.5,null,1,100.0
2019-10-01,1,1,36.0,601.0,1.333333,2,-50.0
2022-03-01,1,1,60.0,2114.0,1.333333,1,0.0
2022-04-01,3,1,57.67,3096.67,1.666667,1,200.0
2022-06-01,3,1,57.67,1721.33,2.333333,3,0.0
2022-07-01,4,1,58.25,3828.75,3.333333,3,33.33
2022-08-01,6295,1,55.44,2238.19,2100.666667,4,157275.0
2022-09-01,9545,42,58.75,2184.96,5281.333333,6295,51.63


### Pola Hari dalam Minggu

Agregasi berikut menyiapkan pola musiman mingguan yang akan digunakan pada visualisasi tahap kedua.

In [19]:
nama_hari = {
    1: "Senin", 2: "Selasa", 3: "Rabu", 4: "Kamis",
    5: "Jumat", 6: "Sabtu", 7: "Minggu",
}

weekday = (
    df
    .with_columns(pl.col("date").dt.weekday().alias("nomor_hari"))
    .group_by("nomor_hari")
    .agg(
        pl.len().alias("jumlah_artikel"),
        pl.col("text_length").mean().round(2).alias("rata_panjang_teks"),
    )
    .sort("nomor_hari")
    .collect()
    .with_columns(
        pl.col("nomor_hari").replace_strict(nama_hari).alias("hari")
    )
)
weekday

nomor_hari,jumlah_artikel,rata_panjang_teks,hari
i8,u32,f64,str
1,150963,2322.99,"""Senin"""
2,155745,2328.44,"""Selasa"""
3,152386,2325.47,"""Rabu"""
4,149673,2320.97,"""Kamis"""
5,142436,2318.51,"""Jumat"""
6,92268,2366.73,"""Sabtu"""
7,84611,2384.4,"""Minggu"""


### Catatan Tahap 1

Tahap pertama menghasilkan tabel bulanan dengan moving average tiga bulan dan pertumbuhan terhadap bulan sebelumnya. Agregasi hari dalam minggu juga telah disiapkan untuk analisis musiman pada tahap visualisasi.

## Bagian 3: Enam Visualisasi Interaktif

Visualisasi dibangun dengan Plotly. Oktober 2023 hanya berisi data tanggal 1, sehingga dikeluarkan dari grafik tren bulanan agar tidak menimbulkan kesimpulan penurunan yang keliru.

In [20]:
monthly_complete = monthly.filter(pl.col("date") < pl.date(2023, 10, 1))

eda_sample = (
    df
    .select(
        "date", "source_domain", "title_length", "text_length",
        "title_length_clean", "text_length_clean",
    )
    .collect()
    .sample(n=20_000, seed=2023103703)
)

top_domains = (
    df
    .group_by("source_domain")
    .agg(pl.len().alias("jumlah_artikel"))
    .sort("jumlah_artikel", descending=True)
    .head(15)
    .collect()
)

domain_metrics = (
    df
    .group_by("source_domain")
    .agg(
        pl.len().alias("jumlah_artikel"),
        pl.col("title_length").mean().round(2).alias("rata_panjang_judul"),
        pl.col("text_length").mean().round(2).alias("rata_panjang_teks"),
        (pl.col("text_length_outlier").mean() * 100).round(2).alias("persen_outlier_teks"),
    )
    .sort("jumlah_artikel", descending=True)
    .head(30)
    .collect()
)

print(f"Sampel distribusi: {eda_sample.height:,} artikel")
print(f"Domain untuk analisis multidimensi: {domain_metrics.height}")

Sampel distribusi: 20,000 artikel
Domain untuk analisis multidimensi: 30


### Visualisasi 1: Tren Artikel Bulanan dan Moving Average

In [21]:
fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=monthly_complete["date"],
    y=monthly_complete["jumlah_artikel"],
    mode="lines+markers",
    name="Jumlah artikel",
    line=dict(color="#2563EB"),
))
fig1.add_trace(go.Scatter(
    x=monthly_complete["date"],
    y=monthly_complete["moving_average_3_bulan"],
    mode="lines",
    name="Moving average 3 bulan",
    line=dict(color="#DC2626", width=3),
))
fig1.update_layout(
    title="Tren Publikasi Artikel per Bulan",
    xaxis_title="Bulan",
    yaxis_title="Jumlah artikel",
    template="plotly_white",
    hovermode="x unified",
    xaxis=dict(rangeslider=dict(visible=True)),
)
fig1.show(config={"displaylogo": False})

### Visualisasi 2: Distribusi Panjang Isi Artikel

In [22]:
fig2 = px.histogram(
    eda_sample,
    x="text_length",
    nbins=80,
    marginal="box",
    log_y=True,
    title="Distribusi Panjang Isi Artikel (Sampel 20.000 Artikel)",
    labels={"text_length": "Panjang isi (karakter)", "count": "Jumlah artikel"},
    color_discrete_sequence=["#0F766E"],
    template="plotly_white",
)
fig2.update_layout(yaxis_title="Jumlah artikel (skala log)")
fig2.show(config={"displaylogo": False})

### Visualisasi 3: Korelasi Variabel Panjang Artikel

In [23]:
numeric_columns = [
    "title_length", "text_length",
    "title_length_clean", "text_length_clean",
]
correlation_values = [
    [eda_sample.select(pl.corr(row, column)).item() for column in numeric_columns]
    for row in numeric_columns
]

fig3 = go.Figure(go.Heatmap(
    z=correlation_values,
    x=numeric_columns,
    y=numeric_columns,
    zmin=-1,
    zmax=1,
    colorscale="RdBu_r",
    text=[[f"{value:.2f}" for value in row] for row in correlation_values],
    texttemplate="%{text}",
    hovertemplate="%{y} vs %{x}<br>Korelasi=%{z:.3f}<extra></extra>",
))
fig3.update_layout(title="Matriks Korelasi Panjang Judul dan Isi", template="plotly_white")
fig3.show(config={"displaylogo": False})

### Visualisasi 4: Komposisi Domain Sumber Berita

In [24]:
fig4 = px.bar(
    top_domains.sort("jumlah_artikel"),
    x="jumlah_artikel",
    y="source_domain",
    orientation="h",
    title="15 Domain dengan Artikel Terbanyak",
    labels={"jumlah_artikel": "Jumlah artikel", "source_domain": "Domain"},
    color="jumlah_artikel",
    color_continuous_scale="Blues",
    template="plotly_white",
)
fig4.update_layout(coloraxis_showscale=False)
fig4.show(config={"displaylogo": False})

### Visualisasi 5: Pola Publikasi Berdasarkan Hari

In [25]:
urutan_hari = ["Senin", "Selasa", "Rabu", "Kamis", "Jumat", "Sabtu", "Minggu"]
fig5 = px.bar(
    weekday,
    x="hari",
    y="jumlah_artikel",
    color="rata_panjang_teks",
    category_orders={"hari": urutan_hari},
    title="Jumlah Artikel dan Rata-rata Panjang Teks per Hari",
    labels={
        "hari": "Hari",
        "jumlah_artikel": "Jumlah artikel",
        "rata_panjang_teks": "Rata-rata panjang teks",
    },
    color_continuous_scale="Viridis",
    template="plotly_white",
)
fig5.show(config={"displaylogo": False})

### Visualisasi 6: Analisis Multidimensi Antar-Domain

In [26]:
fig6 = px.scatter(
    domain_metrics,
    x="rata_panjang_judul",
    y="rata_panjang_teks",
    size="jumlah_artikel",
    color="persen_outlier_teks",
    hover_name="source_domain",
    size_max=55,
    title="Karakteristik 30 Domain Berita Terbesar",
    labels={
        "rata_panjang_judul": "Rata-rata panjang judul",
        "rata_panjang_teks": "Rata-rata panjang isi",
        "jumlah_artikel": "Jumlah artikel",
        "persen_outlier_teks": "Outlier teks (%)",
    },
    color_continuous_scale="Turbo",
    template="plotly_white",
)
fig6.show(config={"displaylogo": False})

### Catatan Tahap 2

Enam visualisasi interaktif telah mencakup tren waktu, distribusi, korelasi, komposisi kategori, pola musiman mingguan, dan korelasi multidimensi antar-domain. Temuan kuantitatifnya dirumuskan pada tahap ketiga.

## Bagian 4: Rumusan Lima Insight Analitik Utama

Metrik berikut dihitung langsung dari dataset lengkap untuk mendukung interpretasi visual dan menghindari kesimpulan yang hanya berdasarkan pengamatan grafik.

In [27]:
monthly_2023 = monthly_complete.filter(pl.col("date").dt.year() == 2023)
januari_2023 = monthly_2023.filter(pl.col("date").dt.month() == 1).item(0, "jumlah_artikel")
agustus_2023 = monthly_2023.filter(pl.col("date").dt.month() == 8).item(0, "jumlah_artikel")
kenaikan_jan_agu = round((agustus_2023 - januari_2023) / januari_2023 * 100, 2)
bulan_puncak = monthly_2023.sort("jumlah_artikel", descending=True).row(0, named=True)

temporal_metrics = pl.DataFrame({
    "metrik": ["Artikel Januari 2023", "Artikel Agustus 2023", "Kenaikan Januari-Agustus (%)", "Bulan puncak"],
    "nilai": [str(januari_2023), str(agustus_2023), str(kenaikan_jan_agu), str(bulan_puncak["date"])],
})
temporal_metrics

metrik,nilai
str,str
"""Artikel Januari 2023""","""61296"""
"""Artikel Agustus 2023""","""117163"""
"""Kenaikan Januari-Agustus (%)""","""91.14"""
"""Bulan puncak""","""2023-08-01"""


In [28]:
distribution_metrics = df.select(
    pl.col("text_length").quantile(0.25).alias("q1_panjang_teks"),
    pl.col("text_length").median().alias("median_panjang_teks"),
    pl.col("text_length").quantile(0.75).alias("q3_panjang_teks"),
    pl.col("text_length").quantile(0.95).alias("p95_panjang_teks"),
    (pl.col("text_length_outlier").mean() * 100).round(2).alias("persen_outlier_teks"),
    pl.corr("title_length", "text_length").round(3).alias("korelasi_judul_teks"),
).collect()
distribution_metrics

q1_panjang_teks,median_panjang_teks,q3_panjang_teks,p95_panjang_teks,persen_outlier_teks,korelasi_judul_teks
f64,f64,f64,f64,f64,f64
1484.0,2133.0,2894.0,4730.0,3.53,0.125


In [29]:
total_artikel = data_summary.item(0, "jumlah_baris")
top_5_domain = top_domains.head(5)
domain_metrics_summary = pl.DataFrame({
    "metrik": [
        "Domain terbesar",
        "Pangsa domain terbesar (%)",
        "Pangsa lima domain terbesar (%)",
    ],
    "nilai": [
        top_domains.item(0, "source_domain"),
        str(round(top_domains.item(0, "jumlah_artikel") / total_artikel * 100, 2)),
        str(round(top_5_domain["jumlah_artikel"].sum() / total_artikel * 100, 2)),
    ],
})
domain_metrics_summary

metrik,nilai
str,str
"""Domain terbesar""","""tribunnews.com"""
"""Pangsa domain terbesar (%)""","""11.62"""
"""Pangsa lima domain terbesar (%…","""40.66"""


In [30]:
weekday_average = weekday.filter(pl.col("nomor_hari") <= 5)["jumlah_artikel"].mean()
weekend_average = weekday.filter(pl.col("nomor_hari") >= 6)["jumlah_artikel"].mean()
hari_tertinggi = weekday.sort("jumlah_artikel", descending=True).row(0, named=True)
hari_terendah = weekday.sort("jumlah_artikel").row(0, named=True)

weekday_metrics = pl.DataFrame({
    "metrik": ["Rata-rata hari kerja", "Rata-rata akhir pekan", "Selisih relatif (%)", "Hari tertinggi", "Hari terendah"],
    "nilai": [
        str(round(weekday_average, 1)),
        str(round(weekend_average, 1)),
        str(round((weekday_average / weekend_average - 1) * 100, 2)),
        hari_tertinggi["hari"],
        hari_terendah["hari"],
    ],
})
weekday_metrics

metrik,nilai
str,str
"""Rata-rata hari kerja""","""150240.6"""
"""Rata-rata akhir pekan""","""88439.5"""
"""Selisih relatif (%)""","""69.88"""
"""Hari tertinggi""","""Selasa"""
"""Hari terendah""","""Minggu"""


### 1. Tren Waktu dan Musiman

Volume artikel meningkat dari **61.296 artikel pada Januari 2023** menjadi **117.163 artikel pada Agustus 2023**, atau naik sekitar **91,14%**. Setelah puncak Agustus, volume September turun 10,37% menjadi 105.019 artikel. Oktober tidak digunakan untuk menilai tren karena data hanya tersedia sampai tanggal 1. Pola ini menunjukkan ekspansi cakupan data sepanjang 2023, tetapi perubahan volume juga dapat dipengaruhi oleh cakupan sumber dataset dan tidak otomatis merepresentasikan pertumbuhan seluruh industri berita.

### 2. Distribusi dan Anomali

Median panjang isi adalah **2.133 karakter**, dengan 50% artikel berada di antara **1.484 dan 2.894 karakter**. Persentil ke-95 mencapai 4.730 karakter dan sekitar **3,53%** artikel ditandai sebagai outlier panjang. Karena artikel panjang masih dapat valid, proses cleaning mempertahankan teks asli dan hanya melakukan winsorization pada fitur numeriknya.

### 3. Korelasi Kunci

Korelasi antara panjang judul dan panjang isi hanya sekitar **0,125**, sehingga hubungan keduanya sangat lemah. Judul yang panjang bukan indikator kuat bahwa isi artikel juga panjang. Analisis atau pemodelan berikutnya sebaiknya memakai kedua fitur secara terpisah dan tidak menjadikan panjang judul sebagai pengganti kompleksitas isi.

### 4. Perilaku Domain dan Konsentrasi Sumber

`tribunnews.com` merupakan sumber terbesar dengan **107.817 artikel** atau **11,62%** dari dataset. Lima domain terbesar secara bersama menyumbang **40,66%** artikel. Konsentrasi ini berarti insight agregat cukup dipengaruhi portal berproduksi tinggi; perbandingan antar-domain perlu menggunakan proporsi atau rata-rata, bukan hanya jumlah absolut. Tribunnews juga memiliki rata-rata isi sekitar 3.391 karakter dan tingkat outlier teks 10,83%, lebih tinggi daripada banyak domain besar lainnya.

### 5. Rekomendasi Praktis Berdasarkan Siklus Mingguan

Rata-rata volume per hari kerja adalah sekitar **150.241 artikel**, sedangkan akhir pekan sekitar **88.440 artikel**, sehingga hari kerja lebih tinggi sekitar **69,88%**. Selasa menjadi hari dengan volume tertinggi dan Minggu terendah. Untuk pemantauan isu, pengumpulan data, atau kebutuhan komputasi, kapasitas terbesar sebaiknya dialokasikan pada hari kerja—khususnya Selasa—sementara akhir pekan dapat digunakan untuk proses batch, evaluasi kualitas, atau pemeliharaan pipeline.

## Keterbatasan Analisis

- Dataset tidak memiliki koordinat geografis, sehingga visualisasi keenam menggunakan analisis multidimensi antar-domain sebagai pengganti analisis spasial.
- Data sebelum 2023 relatif jarang dan Oktober 2023 tidak lengkap.
- Banyaknya artikel per domain mencerminkan isi dataset, bukan ukuran pasar portal berita secara keseluruhan.
- Korelasi tidak menunjukkan hubungan sebab-akibat.

## Validasi Akhir Milestone 3

In [31]:
figures = [fig1, fig2, fig3, fig4, fig5, fig6]
assert len(figures) == 6, "Jumlah visualisasi harus enam."
assert all(len(figure.data) > 0 for figure in figures), "Ada visualisasi tanpa data."
assert monthly.height > 0 and weekday.height == 7, "Agregasi temporal tidak lengkap."
assert data_summary.item(0, "jumlah_baris") == 928_082, "Jumlah baris berbeda dari hasil cleaning."

validation_summary = pl.DataFrame({
    "komponen": ["Dataset bersih", "Operasi temporal", "Visualisasi interaktif", "Insight analitik"],
    "hasil": ["928.082 baris", "group_by_dynamic, rolling_mean, shift", "6 visualisasi", "5 insight"],
    "status": ["LULUS", "LULUS", "LULUS", "LULUS"],
})
validation_summary

komponen,hasil,status
str,str,str
"""Dataset bersih""","""928.082 baris""","""LULUS"""
"""Operasi temporal""","""group_by_dynamic, rolling_mean…","""LULUS"""
"""Visualisasi interaktif""","""6 visualisasi""","""LULUS"""
"""Insight analitik""","""5 insight""","""LULUS"""


## Kesimpulan

Milestone 3 telah menerapkan analisis temporal dan window functions, menghasilkan enam visualisasi interaktif, serta merumuskan lima insight berbasis metrik. Hasil menunjukkan pertumbuhan volume selama sebagian besar 2023, distribusi panjang teks yang menceng ke kanan, hubungan lemah antara panjang judul dan isi, konsentrasi artikel pada beberapa domain besar, dan aktivitas publikasi yang lebih tinggi pada hari kerja.